In [1]:
import os
import sys 
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
load_dotenv(override=True)
ROOT = Path.cwd().parent.parent.parent.resolve()

print(f"ROOT: {ROOT}")
sys.path.append(str(ROOT))
src_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))

if src_path not in sys.path:
    sys.path.append(src_path)

ROOT: /users/oshan/Dev/financial-document-based-agent-system


In [2]:
from src.finchat.main import ExTrRAGQA
from src.cgcore.vectordb.milvus import MilvusDB
from src.cgcore.embedder.openai import OpenAIEmbedder
from src.cgcore.llm.openai import OpenAILlm

from src.cgcore.configs.vectordb.milvus import MilvusConfig
from src.cgcore.configs.embedder.openai import OpenAIEmbedderConfig
from src.cgcore.configs.llm.openai import OpenAILlmConfig

In [3]:
llm_config = OpenAILlmConfig(api_key=os.getenv('OPENAI_API_KEY'))
embedder_config = OpenAIEmbedderConfig(api_key=os.getenv('OPENAI_API_KEY'), model='text-davinci-003', dimesion=os.getenv('MONGO_DB_DIMENSION'))
vectordb_config = MilvusConfig(
                collection_name=os.getenv('MILVUS_COLLECTION_NAME'),
                dimensions=1536,  # Set explicit dimension value
                )


In [4]:
openai = OpenAILlm(llm_config)
embeder = OpenAIEmbedder(embedder_config)
vectordb = MilvusDB(vectordb_config)

MilvusClient connected.
pymilvus ORM connected to localhost:19530 for setup.
Collection 'financial_documents_backend_test' already exists. Skipping creation.


/tmp/ipykernel_1224119/1873515904.py:1: UserWarning: Parameters {'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  openai = OpenAILlm(llm_config)
/users/oshan/Dev/financial-document-based-agent-system/.venv/lib/python3.12/site-packages/langchain_openai/embeddings/base.py:313: UserWarning: WARNING! encoding_format is not default parameter.
                    encoding_format was transferred to model_kwargs.
                    Please confirm that encoding_format is what you intended.
  warnings.warn(


In [5]:
extr_rag = ExTrRAGQA(
    llm=openai,
    embedder=embeder,
    db=vectordb,
    memory="none",
    history=True
)

/users/oshan/Dev/financial-document-based-agent-system/src/finchat/main.py:45: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  self.context = ConversationBufferWindowMemory(k=5)


In [6]:
await extr_rag.chat("If the Bank had recognized this interest income in March 2024, what would the actual profit have been instead of the reported Rs. 516,469 thousand? in peoples bank")

ic| f"Current Context: {current_context}": 'Current Context: '
/users/oshan/Dev/financial-document-based-agent-system/.venv/lib/python3.12/site-packages/langchain_openai/chat_models/base.py:2067: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


ic| ruling: Rule(decision=False, related_context='In March 2024, the Bank reported a profit of Rs. 516,469 thousand.', extra_questions=['What was the amount of interest income that the Bank failed to recognize in March 2024?', 'What is the impact of recognizing this interest income on the reported profit?'])


Decision: Retrieving external documents...
[DEBUG FULL CONTENT] ID: 463710524979398806
[DEBUG FULL CONTENT] Content: '## Reporting Institutions as at 31st December 2011'
[DEBUG FULL CONTENT] Content length: 50
[DEBUG FULL CONTENT] Metadata: {'url': '/users/oshan/Dev/financial-document-based-agent-system/FIU_AR_2011.pdf', 'processed_path': '/users/oshan/Dev/financial-document-based-agent-system/FIU_AR_2011.md', 'parser': 'dockling_remote', 'Header 2': 'Reporting Institutions as at 31st December 2011', 'questions': ['What is the significance of reporting institutions as at 31st December 2011?', 'What are the key characteristics of reporting institutions as at 31st December 2011?', 'How are reporting institutions identified as at 31st December 2011?'], 'original_chunk_id': 'c1bc75c34b5590f0a31f16c7a674e93c0e342d49958d5fddc4f72ce6a467da7a_85'}
[DEBUG] Formatted chunk: content=## Reporting Institutions as at 31st December 2011..., distance=0.37793782353401184
[DEBUG FULL CONTENT] ID: 463710

ic| updated_docs: [{'_id': '463710524979398806',
                    'content': '## Reporting Institutions as at 31st December 2011',
                    'distance': 0.37793782353401184,
                    'meta_data': {'Header 2': 'Reporting Institutions as at 31st December 2011',
                                  'original_chunk_id': 'c1bc75c34b5590f0a31f16c7a674e93c0e342d49958d5fddc4f72ce6a467da7a_85',
                                  'parser': 'dockling_remote',
                                  'processed_path': '/users/oshan/Dev/financial-document-based-agent-system/FIU_AR_2011.md',
                                  'questions': ['What is the significance of reporting '
                                                'institutions as at 31st December 2011?',
                                                'What are the key characteristics of reporting '
                                                'institutions as at 31st December 2011?',
                                    

'To calculate the actual profit of the Bank if the interest income had been recognized in March 2024, we need to consider the interest income amount. Unfortunately, the provided context does not include information about the interest income amount. Therefore, without this crucial piece of information, it is not possible to determine the actual profit of the Bank.'

In [14]:
await extr_rag.chat("The gross interest income refrained from recognition was Rs. 7,955,664 thousand, but the net impact to profit was only Rs. 4,102,118 thousand. What explains this difference?")

ic| f"Current Context: {current_context}": 'Current Context: '
/users/oshan/Dev/financial-document-based-agent-system/.venv/lib/python3.12/site-packages/langchain_openai/chat_models/base.py:2067: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


ic| ruling: Rule(decision=True, related_context='The difference between the gross interest income refrained from recognition and the net impact to profit is due to the deduction of interest expense and other related costs from the gross interest income.', extra_questions=[])


Decision: Sufficient context found in memory.


'The difference between the gross interest income refrained from recognition of Rs. 7,955,664 thousand and the net impact to profit of Rs. 4,102,118 thousand is due to the deduction of interest expense and other related costs from the gross interest income.'

In [16]:
await extr_rag.chat("People's Bank shows extremely strong liquidity ratios (340% LCR Rupee) but also has significant short-term funding needs. Investigate this apparent contradiction.")

ic| f"Current Context: {current_context}": 'Current Context: '
/users/oshan/Dev/financial-document-based-agent-system/.venv/lib/python3.12/site-packages/langchain_openai/chat_models/base.py:2067: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


ic| ruling: Rule(decision=False, related_context="People's Bank shows extremely strong liquidity ratios (340% LCR Rupee) but also has significant short-term funding needs.", extra_questions=[])


Decision: Retrieving external documents...
[DEBUG FULL CONTENT] ID: 462951351664457716
[DEBUG FULL CONTENT] Content: '| Total Stock of High Quality Liquid Assets [Rs.000]          | 1,529,970,594                | 1,417,850,575              | -                            | -                          |
| Liquidity Coverage Ratio - Rupee                            | 340.28                       | 353.86                     | -                            | -                          |
| Liquidity Coverage Ratio - All Currency                     | 282.91                       | 279.52                     | -                            | -                          |
| Net Stable Funding Ratio  (Min. requirement - 100%)         | 180.43                       | 178.92                     | -                            | -                          |
| Asset Quality ( Quality of Loan Portfolio)                  |                              |                            |                       

ic| updated_docs: [{'_id': '462951351664457716',
                    'content': '| Total Stock of High Quality Liquid Assets [Rs.000]          | '
                               '1,529,970,594                | 1,417,850,575              | '
                               '-                            | -                          |
                  '
                               '| Liquidity Coverage Ratio - Rupee                            | '
                               '340.28                       | 353.86                     | '
                               '-                            | -                          |
                  '
                               '| Liquidity Coverage Ratio - All Currency                     | '
                               '282.91                       | 279.52                     | '
                               '-                            | -                          |
                  '
                               '| Net S

"The People's Bank shows an extremely strong Liquidity Coverage Ratio (LCR) in Rupee at 340.28%, indicating that it has a high level of high-quality liquid assets to cover its short-term obligations. However, despite this strong liquidity position, the bank also has significant short-term funding needs, which suggests that it may be relying heavily on short-term funding sources to meet its obligations. This apparent contradiction could indicate that while the bank has a strong liquidity position currently, it may face challenges in the future if it is unable to secure sufficient funding to support its operations."

In [17]:
await extr_rag.chat("Analyze the deposit base composition. What percentage are demand vs savings vs fixed deposits in domestic currency?")

ic| f"Current Context: {current_context}": 'Current Context: '
/users/oshan/Dev/financial-document-based-agent-system/.venv/lib/python3.12/site-packages/langchain_openai/chat_models/base.py:2067: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


ic| ruling: Rule(decision=False, related_context='Context is empty', extra_questions=[])


Decision: Retrieving external documents...
[DEBUG FULL CONTENT] ID: 462951351664457670
[DEBUG FULL CONTENT] Content: '| Demand deposits (current accounts) | 90,812,662                       | 106,000,154                    | 88,381,159                       | 104,707,196                    |
| Savings deposits                   | 861,669,187                      | 837,758,900                    | 863,691,377                      | 843,003,154                    |
| Fixed deposits                     | 1,704,860,535                    | 1,592,808,586                  | 1,817,350,509                    | 1,680,581,482                  |
| Others                             | 2,028,018                        | 1,753,758                      | 3,473,130                        | 3,230,045                      |
| Sub total                          | 2,659,370,402                    | 2,538,321,398                  | 2,772,896,175                    | 2,631,521,877                  |'
[DEBUG

ic| updated_docs: [{'_id': '462951351664457670',
                    'content': '| Demand deposits (current accounts) | '
                               '90,812,662                       | '
                               '106,000,154                    | '
                               '88,381,159                       | '
                               '104,707,196                    |
                  '
                               '| Savings deposits                   | '
                               '861,669,187                      | '
                               '837,758,900                    | '
                               '863,691,377                      | '
                               '843,003,154                    |
                  '
                               '| Fixed deposits                     | '
                               '1,704,860,535                    | '
                               '1,592,808,586                  | '
                

'To analyze the deposit base composition in domestic currency, we need to focus on the components related to domestic currency in the provided data. The breakdown is as follows:\n\n- Demand deposits (current accounts): 2,033,074\n- Savings deposits: 34,883,796\n- Fixed deposits: 279,435,358\n\nTo calculate the percentage of each deposit type in domestic currency, we can use the formula:\n\nPercentage = (Amount of specific deposit type / Total domestic currency deposits) * 100\n\nCalculating for each deposit type:\n\n- Demand deposits percentage: (2,033,074 / 316,382,796) * 100 ≈ 0.64%\n- Savings deposits percentage: (34,883,796 / 316,382,796) * 100 ≈ 11.02%\n- Fixed deposits percentage: (279,435,358 / 316,382,796) * 100 ≈ 88.34%\n\nTherefore, the percentage composition of demand deposits is approximately 0.64%, savings deposits is around 11.02%, and fixed deposits is about 88.34% in domestic currency.'

In [10]:
extr_rag.chat("How much did XYZ Company pay in dividends during 2001?")

<coroutine object ExTrRAGQA.chat at 0xe37098b5f040>

In [11]:
extr_rag.chat("What is the reporting threshold for Cash Transaction Reports (CTRs) and Electronic Fund Transfers (EFTs) that was revised in June 2008?")

<coroutine object ExTrRAGQA.chat at 0xe37098b5f1c0>

In [12]:
extr_rag.chat("List the four categories of expenses mentioned in the profit or loss statement.")

<coroutine object ExTrRAGQA.chat at 0xe37098b5f340>

In [13]:
extr_rag.chat("What is the formula for the accounting equation mentioned in the statement of financial position section?")

<coroutine object ExTrRAGQA.chat at 0xe37098b5f4c0>